# 03 — Results Analysis
**Project:** Optimizer and Learning Rate Study of Ovarian Cancer Prediction  
**Course:** HI 192 — Knowledge Representation and Health Decision Support

This notebook analyses the aggregated results from all 84 simulation runs. Run `02_experiments.ipynb` first to generate `results/summary/all_runs_summary.csv`.

In [ ]:
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RESULTS_DIR  = pathlib.Path('../results')
SUMMARY_CSV  = RESULTS_DIR / 'summary' / 'all_runs_summary.csv'

ARCHITECTURES  = ['VGG19', 'EfficientNetB3', 'ResNet50', 'DenseNet121']
OPTIMIZERS     = ['Adam', 'Adagrad', 'Adamax', 'AdaDelta', 'SGD', 'RMSProp', 'Nadam']
LEARNING_RATES = [1e-4, 1e-5, 1e-6]
LR_LABELS      = ['1e-4', '1e-5', '1e-6']

---
## 1. Load and Display Master Results Table

Load the consolidated metrics from all 84 runs and display the full leaderboard.

In [ ]:
df = pd.read_csv(SUMMARY_CSV)
df = df.sort_values('auc', ascending=False).reset_index(drop=True)
df.index += 1  # 1-based rank

print(f"Total runs loaded: {len(df)}")
pd.set_option('display.max_rows', 90)
pd.set_option('display.float_format', '{:.4f}'.format)
display(df)

---
## 2. Best Model per Architecture

For each of the 4 base architectures, identify the single best-performing configuration (optimizer + LR) by AUC-ROC.

In [ ]:
# TODO: Group by architecture, select row with max AUC
best_per_arch = (
    df.reset_index(names='rank')
    .sort_values('auc', ascending=False)
    .groupby('architecture', sort=False)
    .first()
    .reset_index()
    [['architecture', 'optimizer', 'learning_rate', 'accuracy', 'precision',
      'recall', 'specificity', 'f1', 'auc']]
    .sort_values('auc', ascending=False)
)
print("Best model per architecture (ranked by AUC):")
display(best_per_arch)

---
## 3. Best Model per Optimizer

Identify the best-performing architecture + LR for each optimizer.

In [ ]:
# TODO: Group by optimizer, select row with max AUC
best_per_opt = (
    df.reset_index(names='rank')
    .sort_values('auc', ascending=False)
    .groupby('optimizer', sort=False)
    .first()
    .reset_index()
    [['optimizer', 'architecture', 'learning_rate', 'accuracy', 'precision',
      'recall', 'specificity', 'f1', 'auc']]
    .sort_values('auc', ascending=False)
)
print("Best model per optimizer (ranked by AUC):")
display(best_per_opt)

---
## 4. Best Model per Learning Rate

Identify the best-performing architecture + optimizer for each learning rate.

In [ ]:
# TODO: Group by learning_rate, select row with max AUC
best_per_lr = (
    df.reset_index(names='rank')
    .sort_values('auc', ascending=False)
    .groupby('learning_rate', sort=False)
    .first()
    .reset_index()
    [['learning_rate', 'architecture', 'optimizer', 'accuracy', 'precision',
      'recall', 'specificity', 'f1', 'auc']]
    .sort_values('auc', ascending=False)
)
print("Best model per learning rate (ranked by AUC):")
display(best_per_lr)

---
## 5. Accuracy Heatmap (Optimizer × Learning Rate, one per Architecture)

Four heatmaps — one per architecture — showing mean accuracy for each optimizer × LR combination.

In [ ]:
# TODO: Produce 2×2 grid of accuracy heatmaps
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, arch in enumerate(ARCHITECTURES):
    sub = df[df['architecture'] == arch].copy()
    sub['lr_label'] = sub['learning_rate'].apply(lambda x: f"{float(x):.0e}")
    pivot = sub.pivot_table(index='optimizer', columns='lr_label', values='accuracy', aggfunc='mean')
    pivot = pivot.reindex(index=OPTIMIZERS, columns=LR_LABELS)

    sns.heatmap(
        pivot,
        ax=axes[idx],
        annot=True,
        fmt='.4f',
        cmap='YlGnBu',
        vmin=0,
        vmax=1,
        linewidths=0.5,
        cbar_kws={'shrink': 0.8, 'label': 'Accuracy'},
    )
    axes[idx].set_title(arch, fontweight='bold')
    axes[idx].set_xlabel('Learning Rate')
    axes[idx].set_ylabel('Optimizer')

fig.suptitle('Accuracy Heatmap — Optimizer × Learning Rate by Architecture',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'summary' / 'heatmap_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. AUC Heatmap (Optimizer × Learning Rate, one per Architecture)

In [ ]:
# TODO: Produce 2×2 grid of AUC heatmaps
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, arch in enumerate(ARCHITECTURES):
    sub = df[df['architecture'] == arch].copy()
    sub['lr_label'] = sub['learning_rate'].apply(lambda x: f"{float(x):.0e}")
    pivot = sub.pivot_table(index='optimizer', columns='lr_label', values='auc', aggfunc='mean')
    pivot = pivot.reindex(index=OPTIMIZERS, columns=LR_LABELS)

    sns.heatmap(
        pivot,
        ax=axes[idx],
        annot=True,
        fmt='.4f',
        cmap='RdYlGn',
        vmin=0.5,
        vmax=1.0,
        linewidths=0.5,
        cbar_kws={'shrink': 0.8, 'label': 'AUC-ROC'},
    )
    axes[idx].set_title(arch, fontweight='bold')
    axes[idx].set_xlabel('Learning Rate')
    axes[idx].set_ylabel('Optimizer')

fig.suptitle('AUC-ROC Heatmap — Optimizer × Learning Rate by Architecture',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'summary' / 'heatmap_auc.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Discussion Prompts

Use the following guiding questions to structure the written analysis in `docs/latex/sections/07_discussion.tex`:

1. **Optimizer effect:** Which optimizer achieved the highest mean AUC across all architectures and learning rates? Was there a consistent winner, or did performance vary by architecture?
2. **Adaptive vs. non-adaptive:** Did adaptive optimizers (Adam, Adamax, Adagrad, AdaDelta, RMSProp, Nadam) consistently outperform SGD? At which learning rates did SGD become competitive?
3. **Learning rate sensitivity:** How sensitive was each architecture to the choice of learning rate? Which architectures degraded most with LR = 1e-6?
4. **Architecture ranking:** Which base model produced the best median AUC across all 21 of its runs? Which was the most stable (lowest variance across LR/optimizer combos)?
5. **Overfitting indicators:** Do the accuracy/loss curves show signs of overfitting (training accuracy >> validation accuracy)? How does dropout rate interact with optimizer choice?
6. **Precision vs. Recall trade-off:** In a clinical context (ovarian cancer detection), which is more costly — a false negative (missed cancer) or a false positive (unnecessary biopsy)? Which model configuration best satisfies the clinical priority?
7. **Specificity:** Are there configurations with high recall but low specificity? What does that imply for clinical deployment?
8. **Reproducibility:** How does fixing `RANDOM_SEED = 42` and sharing one data split strengthen the validity of the comparison?